# EEG experiment review: corrected controls and complementary targets

**CPU → Run all. Use Zyion for both Drive prompts.** One notebook runs five phases and reuses completed fits. No local Mac installs, source downloads, NLP downloads, relabeling, or additional full EEG archive.

| Phase | What it does |
|---|---|
|1. Saved inputs and examples|Verify existing feature/label caches; export actual instances for every target label.|
|2. Audit historical results|Reproduce saved split/averaging memberships, quantify repeated source occurrences and identical four-feature/label pairs crossing train/test, and report actual train/test group sizes, including Max40 mean/median/min/max/full40 counts.|
|3. Correct non-EEG controls|One row per source word occurrence, four variables only, independent occurrence/sentence/word splits, fixed logistic C1 and dummy, exact1/10/40/Max40 (binary also5/20). Never count the same occurrence on both sides.|
|4. New EEG prediction targets|Predict word length, relative position and sentence length from saved48regional EEG features. Individual-trial regression: Ridge1, Ridge100, Extra Trees; pre-onset Ridge and median dummy. Familiar, unseen-sentence, unseen-word and unseen-reader splits; MAE, medianAE, R² and correlation. No target metadata is an EEG predictor.|
|5. Revised repetition|Same word in the SAME sentence: first versus second confirmed native fixation. Pair-level reader/sentence/word holdouts, exact1/10/40/Max40, logistic/LDA/pre-onset/four-feature/dummy comparisons. Matched first/second averages contain the same source words.|

## What a corrected non-EEG control can mean
Four features: **word length, word’s relative position, sentence length, control-reading flag**. Audit their actual invariance and per-label control-flag distributions; do not assume control reading is balanced. Previous sentiment controls sometimes had six variables, including duration/next-word delay; their historical numbers remain separately labelled.

A source occurrence is dataset/task/sentence-text key/word index. Features and labels must agree across its reader replicas; conflicting occurrences are excluded and listed rather than silently majority-labelled. Reader holdout is not defined once reader replicas are removed. The new occurrence-level scores are independent benchmarks, **not directly comparable scores on the historical reader-level splits or averages**. Historical familiar/reader results are not retroactively replaced with supposedly identical conditions.

Reader identity cannot have one reader-free row per source word occurrence. It is marked not applicable. Duration varies across readers: the extra duration control uses the **median first-fixation duration across at least3readers per occurrence**, with150/200/250ms cutoffs. This is explicitly a different contextual target, not a replacement score for individual-reader duration. Word identity is closed-set, so there is no unseen-word test for that target. The original broad cross-sentence repetition target is audited for traceability but superseded by phase5.

## First/second fixation preparation
The old feature archive contains only the first stored fixation. Phase5 therefore must read your **existing native TeCo task1.pickle once per unfinished reader** to confirm later fixations. It checks file identity against the validated archive, fixation counts/order/durations, unique waveform location, chronological ordering, same word/sentence, finite in-bounds EEG and reproduction of the saved first-fixation features. A mismatched preparation stops instead of silently mixing pipelines. No minimum duration/amplitude selection is added beyond the original valid first-fixation scope. Only compact48post+12pre features per fixation are retained, not epochs. Paired words stay in one partition; entire training pairs are purged for temporal overlap. Readers without source files/confirmed pairs are explicitly listed. Fewer than6readers yields no partial-cohort classifier.

**Storage:50MB total additional Drive cap**, including compact paired-fixation caches and compressed checkpoints, under `Thesis/Results/word_sentiment_eeg/experiment_review_followup/`. Old results and raw sources stay read-only. No arbitrary package/model bank. Native source reading can be slow; the first run has not been timed. Reader and sentence logs plus30-second heartbeats show progress. A runtime reset reuses completed per-reader pair caches and model fits.

No feature-alone or leave-one-feature-out ablation is included: that step is deferred as requested. Averaging uses known target labels and does not establish single-word deployment accuracy. The data were explored before; no new significance claim is made.


In [ ]:
# Runtime check: run before installation, Drive access, or analysis.
import os
import subprocess

REQUIRED_RUNTIME = 'CPU'
if os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME'):
    raise RuntimeError(f'This notebook needs {REQUIRED_RUNTIME}, not TPU. Choose Runtime > Change runtime type, then rerun all.')
try:
    _gpu_probe = subprocess.run(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15, check=False)
except FileNotFoundError:
    _gpu_names = ''
except subprocess.TimeoutExpired:
    raise RuntimeError('GPU detection timed out. Reconnect the runtime and rerun this cell.') from None
else:
    if _gpu_probe.returncode:
        raise RuntimeError('GPU detection failed. Reconnect the runtime and rerun this cell.')
    _gpu_names = _gpu_probe.stdout.strip()
if REQUIRED_RUNTIME == 'GPU' and not _gpu_names:
    raise RuntimeError('GPU required. Choose Runtime > Change runtime type > T4 GPU, then Run all. No setup or analysis has started.')
if REQUIRED_RUNTIME == 'CPU' and _gpu_names:
    raise RuntimeError('CPU notebook: choose Runtime > Change runtime type > CPU (or hardware accelerator None), then Run all to avoid using GPU quota.')
print('Runtime verified:', _gpu_names if _gpu_names else 'CPU', flush=True)


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/experiment_review_followup'
REPO=Path('/content/word-sentiment-eeg-experiment-review')
CODE_REVISION='ee1902d65cb3803325001aa2cc3e66a0e3bbdae2'
def git(*args):
    result=subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],
                          text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
    if result.returncode:
        raise RuntimeError('Git setup failed:\n'+result.stdout)
    if result.stdout:print(result.stdout,end='',flush=True)
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    fetched=subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD^{commit}'],text=True).strip()
    if fetched!=CODE_REVISION:raise RuntimeError('Fetched code differs from the pinned commit.')
    git('-C',REPO,'checkout','--detach','FETCH_HEAD')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    raise RuntimeError('Code checkout verification failed.')
# Temporary cloud-only scientific environment: no base-runtime upgrades/restart.
# BOOTSTRAP_START: also exercised verbatim in cloud CI.
# Use a file-rendering backend in subprocesses, not Colab's unavailable inline plugin.
os.environ['MPLBACKEND']='Agg'
ENV=Path('/content/word-eeg-paper-classification-python')
PYTHON=ENV/'bin/python'
def setup_command(command):
    print('SETUP:', ' '.join(map(str,command)), flush=True)
    result=subprocess.run(list(map(str,command)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    if result.stdout:print(result.stdout,flush=True)
    if result.returncode:
        raise RuntimeError('Scientific environment setup failed; the full diagnostic is printed above. No EEG processing started.')
# Colab may lack ensurepip. This also repairs a partially created environment.
setup_command([sys.executable,'-m','venv','--without-pip',str(ENV)])
setup_command([sys.executable,'-m','pip','--python',str(PYTHON),'install',
               '--only-binary=:all:','-r',str(REPO/'requirements-paper-classification.txt')])
setup_command([sys.executable,'-m','pip','--python',str(PYTHON),'check'])
setup_command([str(PYTHON),'-c',
    'import numpy,pandas,scipy,sklearn,matplotlib; '
    'print("Scientific environment ready:",numpy.__version__,pandas.__version__,scipy.__version__,sklearn.__version__,matplotlib.__version__)'])
# BOOTSTRAP_END
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# Verify account quota before loading any EEG. Use Zyion for both prompts.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();about=response.json();quota=about['storageQuota']
email=about.get('user',{}).get('emailAddress','unavailable')
print('Authenticated quota account:',email)
if email!='zyionygvmutch39@gmail.com':
    raise RuntimeError('Choose your Zyion account for Drive authentication and mounting; it contains the saved trials.')
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a storage limit; cannot verify free space.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
used=sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) if OUT.exists() else 0
if FREE_DRIVE_BYTES<max(0,50_000_000-used):
    raise RuntimeError('Reserve at most 50 MB for results. Existing caches are reused; the same-word refixation phase reads native sources once and saves compact features only.')
print(f'Free space: {FREE_DRIVE_BYTES/1e9:.2f} GB; new output cap: 50 MB.')
print('New output folder:',OUT)


In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[str(PYTHON),'-u',str(REPO/'scripts/experiment_review_followup.py'),'--thesis-root',str(THESIS),'--free-drive-bytes',str(FREE_DRIVE_BYTES)]
run_with_progress(command,OUT/'runtime_status.json',activity='Experiment review: independent controls, numeric EEG targets and confirmed fixation pairs')


In [ ]:
import hashlib
state=json.loads((OUT/'complete.json').read_text())
assert state['status']=='completed'
for name,sha in state['checksums'].items():
    if hashlib.sha256((OUT/name).read_bytes()).hexdigest()!=sha:raise RuntimeError('Checksum failed: '+name)
print('Computation finished. Paired-fixation coverage:',state['native_pair_scope'])
print('Earlier results are unchanged. Corrected controls use a different occurrence-level evaluation.')
for filename,title in [
    ('corrected_occurrence_controls.csv','Corrected occurrence-level non-EEG controls'),
    ('eeg_numeric_predictions.csv','EEG prediction of numeric word/sentence properties'),
    ('confirmed_fixation_results.csv','First versus second confirmed fixation on the same word'),
    ('audit/historical_occurrence_overlap_and_group_sizes.csv','Historical overlap and exact averaging-size audit'),
    ('label_examples.csv','Actual examples for every label'),
    ('fixation_label_examples.csv','Actual first/second fixation examples'),
    ('control_unavailable.csv','Unavailable or non-comparable controls'),
    ('numeric_unavailable.csv','Unavailable numeric conditions'),
    ('fixation_unavailable.csv','Missing native sources, pair coverage or unavailable classifiers')]:
    path=OUT/filename
    display(Markdown('## '+title))
    if not path.exists():print('Not available; see preceding coverage reports.');continue
    try:frame=pd.read_csv(path)
    except pd.errors.EmptyDataError:print('No rows.');continue
    if 'balanced_accuracy' in frame:
        for col in ['balanced_accuracy','macro_f1','chance']:
            if col in frame:frame[col]=frame[col]*100
    with pd.option_context('display.max_rows',200,'display.max_columns',25,'display.max_colwidth',100):display(frame.round(3))
print('Saved results:',OUT)
print('Balanced accuracy/F1 are percentages in this DISPLAY only; saved CSVs retain0–1 units. Regression MAE uses characters/words/fraction; R² is not accuracy.')
